In [1]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Saving all .csv files in folder to list.
path = "MachineLearningCVE/"
files = [file for file in glob.glob(path + "**/*.csv", recursive=True)]

In [ ]:
[print(f) for f in files]

In [ ]:
# Reading all the csv files into dataframes and putting thoose DFs to one list.

dataset = [pd.read_csv(f) for f in files]

In [ ]:
# shape of the each files

for d in dataset:
    print(d.shape)


In [ ]:
# comparing the columns of each tables

for i in range(0,len(dataset)):
    if i != len(dataset)-1:
        same_columns = dataset[i].columns == dataset[i+1].columns
        
        if False in same_columns:
            print(i)
            break

    same_columns

In [ ]:
# Combining all tables into one dataset

dataset = pd.concat([d for d in dataset]).drop_duplicates(keep=False)
dataset.reset_index(drop=True, inplace = True)

In [ ]:
# Saving combined data

dataset.to_csv("CICIDS2017/Dataset_work.csv", index=False)

In [ ]:
# dataset shape

dataset.shape


In [ ]:
# checking the datatypes of each features
#dataset.dtypes
dataset.info()

In [ ]:
# descriptive statistics of the dataset
dataset.describe()
#dataset.describe().transpose()

In [ ]:
# actual dataset

dataset.head(10)

In [ ]:
# Removing whitespaces in column names.

col_names = [col.replace(' ', '') for col in dataset.columns]
dataset.columns = col_names
dataset.head()

In [ ]:
# Dataset conatains 15 labels.
#print(dataset[' Label'].unique())
#len(dataset[' Label'].unique())

dataset['Label'].value_counts()

In [ ]:
# This next snippet uses regular expressions to replace wierd characters with dunders.

label_names = dataset['Label'].unique()


import re

label_names = [re.sub("[^a-zA-Z ]+", "", l) for l in label_names]
label_names = [re.sub("[\s\s]", '_', l) for l in label_names]
label_names = [lab.replace("__", "_") for lab in label_names]

label_names, len(label_names)

In [ ]:
# Replacing 'Label' column values with new readable values.

labels = dataset['Label'].unique()

for i in range(0,len(label_names)):
    dataset['Label'] = dataset['Label'].replace({labels[i] : label_names[i]})
    
dataset['Label'].unique()

In [ ]:
# changing attack labels to their respective attack class
# total 38 different sub-attacks 
def change_label(dataset):
    dataset.Label.replace(['DoS_slowloris','DoS_Slowhttptest','DoS_Hulk','DoS_GoldenEye','DDoS'],'Dos',inplace=True)
    dataset.Label.replace(['Web_Attack_Brute_Force','Web_Attack_XSS', 'Web_Attack_Sql_Injection','Bot'],'Botnet',inplace=True)
    dataset.Label.replace(['Heartbleed','PortScan'],'PortScan',inplace=True)
    dataset.Label.replace(['Infiltration','FTPPatator','SSHPatator'],'Firewall',inplace=True)


In [ ]:
# calling change_label() function
change_label(dataset)

In [ ]:
# distribution of attack classes
dataset.Label.value_counts()

In [ ]:
# checking null values 

dataset.isnull().values.any()

In [ ]:
 #Checking which column/s contain NULL values.

[col for col in dataset if dataset[col].isnull().values.any()]


In [ ]:
# Checking how many NULL values it this column contains.

dataset['FlowBytes/s'].isnull().sum()

In [ ]:
# Removing rows that contain NULL values

dataset.dropna(inplace=True)

In [ ]:
dataset.isnull().any().any()

In [ ]:
# converting string values to float64 

labl = dataset['Label']
dataset = dataset.loc[:, dataset.columns != 'Label'].astype('float64')

In [ ]:


# Checking if all values are finite.

np.all(np.isfinite(dataset))



In [ ]:


# Checking what column/s contain non-finite values.

nonfinite = [col for col in dataset if not np.all(np.isfinite(dataset[col]))]

nonfinite



In [ ]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['FlowBytes/s']).sum()

dataset.shape[0] - finite

In [ ]:
# Replacing infinite values with NaN values.
dataset = dataset.replace([np.inf, -np.inf], np.nan)

In [ ]:
np.any(np.isnan(dataset))

In [ ]:
dataset = dataset.merge(labl, how='outer', left_index=True, right_index=True)

In [ ]:
dataset.dropna(inplace=True)

In [ ]:
dataset.shape

In [ ]:
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

In [ ]:
# selecting numeric attributes columns from data
numeric_col = dataset.select_dtypes(include='number').columns

# using standard scaler for normalizing
std_scaler = StandardScaler()
def normalization(df,col):
  for i in col:
    arr = df[i]
    arr = np.array(arr)
    df[i] = std_scaler.fit_transform(arr.reshape(len(arr),1))
  return df


In [ ]:
# calling the normalization() function
dataset = normalization(dataset.copy(),numeric_col)

# data after normalization
dataset.head()

In [ ]:
# binary classification ##

In [ ]:
# changing attack category

bin_label = pd.DataFrame(dataset.Label.map(lambda x:'BENIGN' if x=='BENIGN' else 'ATTACK'))

In [ ]:
bin_data = dataset.copy()
bin_data['Label'] = bin_label

In [ ]:
from sklearn.preprocessing import LabelEncoder

LE1 = LabelEncoder()

enc_label = bin_label.apply(LE1.fit_transform)
bin_data['intrusion']= enc_label

In [ ]:
LE1.classes_

In [ ]:
bin_data.head()

In [ ]:
# one-hot-encoding for attack label

bin_data = pd.get_dummies(bin_data,columns=['Label'],prefix="",prefix_sep="")
bin_data['Label']= bin_label
bin_data

In [ ]:
# attack distribution for binary

plt.figure(figsize=(8,8))
plt.pie(bin_data.Label.value_counts(),labels=bin_data.Label.unique(),autopct='%0.2f%%')
plt.title("pie chart distribution of normal and abnormal labels")
plt.legend()
plt.savefig('CICIDS2017/Pie_chart_binary.png')
plt.show()

In [ ]:
## feature extraction ##

In [ ]:
# final data for binary classification ##

In [ ]:
# creating dataframe with only numeric attributes of binary class and encoded label attribute
numeric_col = dataset.select_dtypes(include = 'number').columns

numeric_bin = bin_data[numeric_col]
numeric_bin['intrusion'] = bin_data['intrusion']

In [ ]:
# extracting features which have more than 0.5 correlation (pearson correlation coefficient)

corr=numeric_bin.corr()
corr_y = abs(corr['intrusion'])
highest_corr = corr_y[corr_y >0.5]
highest_corr.sort_values(ascending=True)

In [ ]:
numeric_bin = bin_data[['IdleMean','FwdIATMax','FlowIATMax','IdleMax','FwdIATStd','PacketLengthMean','AveragePacketSize','PacketLengthVariance',
                        'MaxPacketLength','PacketLengthStd','AvgBwdSegmentSize','BwdPacketLengthMean','BwdPacketLengthMax',
                        'BwdPacketLengthStd']]


In [ ]:

bin_data = numeric_bin.join(bin_data[['intrusion','BENIGN','ATTACK','Label']])

In [ ]:
bin_data.to_csv("CICIDS2017/bin_data_balanced.csv")
bin_data

In [ ]:
# counting the attacks for binary

bin_data['intrusion'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score,auc,roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import time
import warnings

In [ ]:
# classifier for binary classification dataset


warnings.filterwarnings("ignore")


X = bin_data.iloc[:,0:14].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = bin_data['intrusion'] # target attribute

X_train, X_validation, Y_train, Y_validation = train_test_split(X, y, test_size=0.20, random_state=1)
# Spot Check Algorithms
models = []
models.append(('LR', LogisticRegression(solver='lbfgs',multi_class='multinomial')))
models.append(('LDA', LinearDiscriminantAnalysis(tol=0.0001)))
models.append(('KNN', KNeighborsClassifier(n_neighbors=3)))
models.append(('CART', DecisionTreeClassifier(max_depth=5)))
models.append(('SVM', SVC(gamma=0.001, max_iter=200)))

print ('Model\tAcc\tpr\tRecall\tF1\tFAR\tsen\tspe\tExecution_time')

# evaluate each model in turn
results = []
names = []

for name, model in models:
    start_time = time.time()
    kfold = StratifiedKFold(n_splits=10, random_state=1, shuffle=True)
    
    cv_results = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='accuracy').mean()
    precision = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='precision').mean()
    recall = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='recall').mean()
    f1score = cross_val_score(model, X_train, Y_train, cv=kfold, scoring='f1_weighted').mean()
    
    m = model.fit(X_train, Y_train)
    predict = m.predict(X_validation)
    cm = confusion_matrix(Y_validation, predict)
    # Creating a dataframe for a array-formatted Confusion matrix,so it will be easy for plotting.
    cm_df = pd.DataFrame(cm)
    
    total1=sum(sum(cm))
    false_alaram_rate = cm[1,0]/(cm[1,0]+cm[0,0])
    sensitivity = cm[1,1]/(cm[1,1]+cm[0,1])
    specificity = cm[0,0]/(cm[1,0]+cm[0,0])
    
    delta = time.time()- start_time
    results.append(cv_results)
    names.append(name)
    print('{}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.3f}\t{:.2f} sec'.format(name, cv_results, precision, recall, f1score, false_alaram_rate, sensitivity, specificity, delta))
    print(cm_df)
      


In [ ]:
#Creating plot to show the ROC for all Models


index = 1
for model in models:
    fp, tp, th = roc_curve(Y_validation, predict)
    roc_auc_mla = auc(fp, tp)
    MLA_name = model.__class__.__name__
    plt.plot(fp, tp, lw=2, alpha=0.3, label='ROC %s (AUC = %0.2f)'  % (MLA_name, roc_auc_mla))
index+=1

plt.title('ROC Curve comparison')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.plot([0,1],[0,1],'r--')
plt.xlim([0,1])
plt.ylim([0,1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')    
plt.show()